# Adam by Hand → Fair Tuning: Colab GPU Execution Pipeline

This notebook executes the GPU-intensive experiments (Parts 3, 4, 5) on a Google Colab GPU runtime (e.g. Nvidia T4),
persisting results idempotently to Google Drive after each individual run.

**Notebook:** `notebooks/adam_by_hand_tuning.ipynb`  
**Repository:** [https://github.com/thinkTinkerGreen/adam_optimiser](https://github.com/thinkTinkerGreen/adam_optimiser)

In [ ]:
# Cell 1: Mount Google Drive for persistent run storage
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/adam-fair-tuning')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ['ADAMLAB_RESULTS_DIR'] = str(DRIVE_ROOT / 'results')
    print(f'Google Drive mounted successfully. Results directory: {os.environ["ADAMLAB_RESULTS_DIR"]}')
except Exception as e:
    print(f'Running in local/offline environment: {e}')
    os.environ.setdefault('ADAMLAB_RESULTS_DIR', 'results')

In [ ]:
# Cell 2: Clone repository (if in Colab), enter repo directory, and install adamlab
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/thinkTinkerGreen/adam_optimiser.git"
COLAB_TARGET = Path("/content/adam_optimiser")

# If running in Colab and not already inside the repository
if not Path("pyproject.toml").exists():
    if not (COLAB_TARGET / "pyproject.toml").exists():
        print(f"Cloning {REPO_URL} into {COLAB_TARGET}...")
        subprocess.run(["git", "clone", REPO_URL, str(COLAB_TARGET)], check=True)
    os.chdir(str(COLAB_TARGET))
    get_ipython().run_line_magic("cd", str(COLAB_TARGET))

# Ensure src/ and repo root are in sys.path
repo_root = Path.cwd().resolve()
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Install lightweight dependencies and editable package
!pip install -q pydantic jinja2
!pip install -e . --no-deps

import adamlab
from adamlab.determinism import get_git_sha

print(f"Current working directory: {Path.cwd()}")
print(f"adamlab version: {adamlab.__version__}, commit: {get_git_sha()}")

In [ ]:
# Cell 3: Environment verification & GPU assertion
import torch
from adamlab.determinism import get_environment_metadata

meta = get_environment_metadata()
print('Runtime Metadata:', meta)
if not torch.cuda.is_available():
    print('WARNING: CUDA not detected! Running on CPU fallback.')
else:
    print(f'Active GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

In [ ]:
# Cell 4: Determinism smoke test on active device
import torch
from adamlab.io import RunSpec
from adamlab.train import run_training

device = 'cuda' if torch.cuda.is_available() else 'cpu'

spec = RunSpec(
    experiment='determinism_smoke',
    width=64,
    depth=2,
    lr=0.001,
    schedule='cosine',
    total_steps=10,
    warmup_steps=2,
    seed=42,
    device=device,
    use_synthetic=True,
    eval_steps=[10],
)
res1 = run_training(spec, save_to_disk=False)
res2 = run_training(spec, save_to_disk=False)
assert res1.final_train_loss == res2.final_train_loss, 'Determinism check failed!'
print(f'Determinism test PASSED on {device}: Bitwise identical loss curves.')

In [ ]:
# Cell 5: Benchmark and estimated full-sweep runtime
import time
import torch
from adamlab.io import RunSpec
from adamlab.train import run_training

device = 'cuda' if torch.cuda.is_available() else 'cpu'
timings = {}
for w in [256, 512, 1024]:
    b_spec = RunSpec(
        experiment='benchmark',
        width=w,
        depth=4,
        lr=0.001,
        schedule='cosine',
        total_steps=20,
        warmup_steps=5,
        seed=0,
        device=device,
        use_synthetic=True,
        eval_steps=[],
    )
    t0 = time.time()
    run_training(b_spec, save_to_disk=False)
    dt = time.time() - t0
    step_sec = dt / 20.0
    timings[w] = step_sec
    print(f'Width {w}: {step_sec*1000:.1f} ms/step ({dt:.2f}s for 20 steps)')

# Estimate full matrix (51 trials/width * 300 steps = 15,300 steps/width)
est_hours = sum(timings[w] * 15300 for w in [256, 512, 1024]) / 3600.0
print(f'Total estimated matrix compute time on {device}: {est_hours:.2f} hours')

In [ ]:
# Cell 6: Run matrix execution loop (idempotent, resumable)
import torch
from scripts.build_run_matrix import generate_run_matrix
from adamlab.train import run_training

device = 'cuda' if torch.cuda.is_available() else 'cpu'
matrix = generate_run_matrix(device=device, use_synthetic=False)
print(f'Processing {len(matrix)} runs on {device} in deterministic sequence...')

for idx, run_spec in enumerate(matrix, start=1):
    print(f'[{idx}/{len(matrix)}] Executing {run_spec.experiment} (width={run_spec.width}, lr={run_spec.lr:.2e}, seed={run_spec.seed})...')
    run_training(run_spec, save_to_disk=True)

In [ ]:
# Cell 7: Sync results from Google Drive to local repository
!mkdir -p results/runs
!cp -ru /content/drive/MyDrive/adam-fair-tuning/results/* results/ 2>/dev/null || true
print('Results synchronized from Google Drive to local repository.')

In [ ]:
# Cell 8: Aggregations, Figures & Final Checks
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
!python -m experiments.p5_lr_sweep --device $device
!python -m experiments.p4_cosine_vs_wsd --device $device
!python -m experiments.p3_update_ratio --device $device
!python -m scripts.make_figures
!python -m scripts.render_readme
!python -m scripts.check_readme
print('All Colab pipeline stages completed successfully!')